# 01_GSE120926_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.2, 4.1.3**

**Reads:** GSE120926 raw 10x matrices and barcode/metadata files (GEO).

**Writes:** GSE120926_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** CD8 selection is marker-based here: PTPRC > 0, then (CD3D > 0 or CD3E > 0), then (CD8A > 0 or CD8B > 0) and CD4 == 0.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


Setup paths

In [ ]:
import os, gzip
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import scipy.sparse as sp
import matplotlib.pyplot as plt

data_dir = f"{DATA_ROOT}/scVI/11 GSE120926"

mtx_path      = os.path.join(data_dir, "GSE120926_NPC_10X_matrix.mtx.gz")
genes_path    = os.path.join(data_dir, "GSE120926_NPC_10X_genes.tsv.gz")
barcodes_path = os.path.join(data_dir, "GSE120926_NPC_10X_barcodes.tsv.gz")

for p in [mtx_path, genes_path, barcodes_path]:
    assert os.path.exists(p), f"Missing: {p}"

print("✅ Files found.")

Load the 10x matrix into AnnData

In [ ]:
# 1) Matrix (genes x cells) in MTX
with gzip.open(mtx_path, "rb") as f:
    X = scipy.io.mmread(f).tocsr()

print("MTX shape (genes x cells):", X.shape, "nnz:", X.nnz)

# 2) Genes/features table
genes = pd.read_csv(genes_path, sep="\t", header=None, compression="gzip")

# Common cases:
# - 2 columns: gene_id, gene_symbol
# - 3 columns (features.tsv): gene_id, gene_symbol, feature_type
if genes.shape[1] >= 2:
    gene_ids = genes.iloc[:, 0].astype(str).values
    gene_symbols = genes.iloc[:, 1].astype(str).values
else:
    raise ValueError("genes.tsv.gz has unexpected format (needs >=2 columns).")

# 3) Barcodes
barcodes = pd.read_csv(barcodes_path, sep="\t", header=None, compression="gzip").iloc[:,0].astype(str).values

# 4) Build AnnData (cells x genes)
adata = sc.AnnData(X.T)  # transpose to cells x genes
adata.obs_names = barcodes
adata.var["gene_ids"] = gene_ids
adata.var_names = gene_symbols

# Make gene symbols unique (important for downstream)
adata.var_names_make_unique()

# Keep raw counts in a layer
adata.layers["counts"] = adata.X.copy()

print(adata)

In [ ]:
# Inspect barcodes to see if they encode sample/patient
print("n_cells:", adata.n_obs)
print("Example barcodes:")
print(adata.obs_names[:10].tolist())
print("Barcode length stats:", pd.Series([len(x) for x in adata.obs_names]).describe())

Load Patient Metadata

In [ ]:
import os
import pandas as pd

meta_path = os.path.join(data_dir, f"{DATA_ROOT}/scVI/11 GSE120926/GSE120926_NPC_10X_cell-barcode-process.txt.gz")

meta = pd.read_csv(meta_path, sep="\t", compression="gzip")

print(meta.head())
print(meta.columns)
print("Meta rows:", meta.shape[0])

Attach metadata to AnnData

In [ ]:
meta = meta.rename(columns={"Unnamed: 0": "barcode"})
meta = meta.set_index("barcode")

# sanity check: do all barcodes match?
print("All barcodes match:", meta.index.equals(pd.Index(adata.obs_names)))

# join into obs
adata.obs = adata.obs.join(meta, how="left")

# quick checks
print(adata.obs[["library", "type"]].isna().sum())
print("Unique libraries:", adata.obs["library"].nunique())
print("Type counts:\n", adata.obs["type"].value_counts())

In [ ]:
cells_per_lib = adata.obs["library"].value_counts().sort_index()

plt.figure(figsize=(12,4))
plt.bar(cells_per_lib.index.astype(str), cells_per_lib.values)
plt.xticks(rotation=90)
plt.ylabel("Number of cells")
plt.title("Cells per library (pre-QC)")
plt.tight_layout()
plt.show()

In [ ]:
type_counts = adata.obs["type"].value_counts()

plt.figure()
plt.bar(type_counts.index.astype(str), type_counts.values)
plt.ylabel("Number of cells")
plt.title("Cells per type (pre-QC)")
plt.show()

In [ ]:
ct = pd.crosstab(adata.obs["library"], adata.obs["type"]).sort_index()

ct.plot(kind="bar", stacked=True, figsize=(12,5))
plt.ylabel("Number of cells")
plt.title("Cells per library (stacked by type, pre-QC)")
plt.tight_layout()
plt.show()

In [ ]:
import os
os.getcwd()

In [ ]:
import os

os.chdir(f"{DATA_ROOT}/scVI/11 GSE120926")
os.getcwd()

In [ ]:
import os
import scanpy as sc

# Ensure counts layer exists (you already set it earlier, but just in case)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

preqc_path = os.path.join(data_dir, "GSE120926_10X_preQC_rawcounts.h5ad")
adata.write_h5ad(preqc_path)
print("✅ Saved pre-QC:", preqc_path)

In [ ]:
import numpy as np

# Flags
adata.var["mt"]   = adata.var_names.str.upper().str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.upper().str.startswith(("RPS","RPL"))
adata.var["hb"]   = adata.var_names.str.upper().str.startswith(("HBA","HBB"))

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt","ribo","hb"],
    percent_top=[50, 100],
    log1p=False,
    inplace=True
)

# Rename to your preferred names (optional, but consistent)
adata.obs.rename(columns={
    "total_counts": "n_counts",
    "n_genes_by_counts": "n_genes",
    "pct_counts_mt": "pct_mt",
    "pct_counts_ribo": "pct_ribo",
    "pct_counts_hb": "pct_hb"
}, inplace=True)

adata.obs[["n_counts","n_genes","pct_mt","pct_ribo","pct_hb"]].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99])

In [ ]:
import matplotlib.pyplot as plt

def hist_log10(x, title, xlabel, bins=60):
    x = np.asarray(x)
    x = x[x > 0]
    plt.figure()
    plt.hist(np.log10(x), bins=bins)
    plt.xlabel(f"log10({xlabel})")
    plt.title(title)
    plt.show()

def hist_lin(x, title, xlabel, bins=60):
    plt.figure()
    plt.hist(x, bins=bins)
    plt.xlabel(xlabel)
    plt.title(title)
    plt.show()

# Overall
hist_log10(adata.obs["n_counts"], "Pre-QC: total counts per cell", "n_counts")
hist_log10(adata.obs["n_genes"],  "Pre-QC: genes per cell", "n_genes")
hist_lin(adata.obs["pct_mt"],     "Pre-QC: % mitochondrial", "pct_mt")

# Scatter
plt.figure()
plt.scatter(adata.obs["n_counts"], adata.obs["n_genes"], s=2)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("n_counts (log)"); plt.ylabel("n_genes (log)")
plt.title("Pre-QC: n_genes vs n_counts")
plt.show()

plt.figure()
plt.scatter(adata.obs["n_counts"], adata.obs["pct_mt"], s=2)
plt.xscale("log")
plt.xlabel("n_counts (log)"); plt.ylabel("pct_mt")
plt.title("Pre-QC: pct_mt vs n_counts")
plt.show()

# By tissue type (tumor vs non)
for t in adata.obs["type"].unique():
    sub = adata.obs["type"] == t
    hist_log10(adata.obs.loc[sub,"n_counts"], f"Pre-QC n_counts ({t})", "n_counts")
    hist_log10(adata.obs.loc[sub,"n_genes"],  f"Pre-QC n_genes ({t})",  "n_genes")
    hist_lin(adata.obs.loc[sub,"pct_mt"],     f"Pre-QC pct_mt ({t})",   "pct_mt")

# By library: median QC per library
med = adata.obs.groupby("library")[["n_counts","n_genes","pct_mt"]].median().sort_index()

plt.figure(figsize=(12,4))
plt.bar(med.index.astype(str), med["n_counts"].values)
plt.xticks(rotation=90); plt.ylabel("Median n_counts")
plt.title("Pre-QC: Median n_counts per library")
plt.tight_layout(); plt.show()

plt.figure(figsize=(12,4))
plt.bar(med.index.astype(str), med["n_genes"].values)
plt.xticks(rotation=90); plt.ylabel("Median n_genes")
plt.title("Pre-QC: Median n_genes per library")
plt.tight_layout(); plt.show()

plt.figure(figsize=(12,4))
plt.bar(med.index.astype(str), med["pct_mt"].values)
plt.xticks(rotation=90); plt.ylabel("Median pct_mt")
plt.title("Pre-QC: Median pct_mt per library")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import pandas as pd

min_genes = 200
min_counts = 500

max_genes  = int(np.quantile(adata.obs["n_genes"], 0.995))
max_counts = int(np.quantile(adata.obs["n_counts"], 0.995))

max_pct_mt = 12

print("QC thresholds:")
print(dict(min_genes=min_genes,
           min_counts=min_counts,
           max_genes=max_genes,
           max_counts=max_counts,
           max_pct_mt=max_pct_mt))

qc_mask = (
    (adata.obs["n_genes"]  >= min_genes) &
    (adata.obs["n_counts"] >= min_counts) &
    (adata.obs["n_genes"]  <= max_genes) &
    (adata.obs["n_counts"] <= max_counts) &
    (adata.obs["pct_mt"]   <= max_pct_mt)
)

print("\nCells before:", adata.n_obs)
print("Cells after :", int(qc_mask.sum()))
print("Fraction kept:", float(qc_mask.mean()))

print("\nBy type:")
print(pd.crosstab(adata.obs["type"], qc_mask))

print("\nBy library (first 10):")
print(pd.crosstab(adata.obs["library"], qc_mask).head(10))

In [ ]:
adata_qc = adata[qc_mask].copy()

# Preserve raw counts
adata_qc.layers["counts"] = adata_qc.layers["counts"]

postqc_path = os.path.join(data_dir, "GSE120926_10X_postQC_scvi_rawcounts.h5ad")
adata_qc.write_h5ad(postqc_path)

print("✅ Saved post-QC object:", postqc_path)
print(adata_qc)

Import scanpy and reconstruct the path

In [ ]:
import scanpy as sc
import os

DATA_DIR = f"{DATA_ROOT}/scVI/11 GSE120926 NPC"

scvi_fp = os.path.join(DATA_DIR, "GSE120926_10X_postQC_scvi_rawcounts.h5ad")

In [ ]:
adata = sc.read_h5ad(scvi_fp)

print("Loaded object:")
print(adata)

In [ ]:
immune_mask = adata[:, "PTPRC"].X > 0  # CD45
GSE120926_immune = adata[immune_mask].copy()

In [ ]:
import numpy as np

immune_mask = (adata[:, "PTPRC"].layers["counts"].toarray().ravel() > 0)
adata_imm = adata[immune_mask].copy()
print(adata_imm.shape)

In [ ]:
cd3d = adata_imm[:, "CD3D"].layers["counts"].toarray().ravel()
cd3e = adata_imm[:, "CD3E"].layers["counts"].toarray().ravel()

t_mask = (cd3d > 0) | (cd3e > 0)
adata_T = adata_imm[t_mask].copy()
print("T cells:", adata_T.n_obs)

In [ ]:
cd8a = adata_T[:, "CD8A"].layers["counts"].toarray().ravel() if "CD8A" in adata_T.var_names else np.zeros(adata_T.n_obs)
cd8b = adata_T[:, "CD8B"].layers["counts"].toarray().ravel() if "CD8B" in adata_T.var_names else np.zeros(adata_T.n_obs)
cd4  = adata_T[:, "CD4"].layers["counts"].toarray().ravel()  if "CD4"  in adata_T.var_names else np.zeros(adata_T.n_obs)

cd8_mask = ((cd8a > 0) | (cd8b > 0)) & (cd4 == 0)
adata_CD8 = adata_T[cd8_mask].copy()
print("CD8 (CD4 excluded):", adata_CD8.n_obs)

In [ ]:
import scanpy as sc
sc.pl.violin(
    adata_CD8,
    ["PTPRC","CD3D","CD3E","CD8A","CD8B","CD4","NKG7","MS4A1","LYZ","EPCAM"],
    groupby="type",
    use_raw=False
)

In [ ]:
adata_CD8.obs["Tissue"] = adata_CD8.obs["type"].replace({"tu":"Tumour", "non":"Normal"})

In [ ]:
print(adata_CD8.obs["Tissue"].value_counts())
print(adata_CD8.obs["library"].nunique())
print(adata_CD8.obs.groupby(["library","Tissue"]).size().head(20))

In [ ]:
adata_CD8.write_h5ad(os.path.join(DATA_DIR, "GSE120926_CD8_postQC_scvi_rawcounts.h5ad"))